# 프로덕션 모니터링: 대규모 자동화된 품질 관리

MLflow의 프로덕션 모니터링은 프로덕션 트래픽의 샘플에 대해 자동으로 품질 평가를 실행하여, 수동 개입 없이도 GenAI 앱이 높은 품질 기준을 유지하도록 보장합니다. MLflow를 사용하면 오프라인 평가를 위해 정의한 동일한 메트릭을 프로덕션 환경에서도 사용할 수 있어, 개발부터 프로덕션까지 전체 애플리케이션 수명 주기에 걸쳐 일관된 품질 평가가 가능합니다.

**주요 이점:**

* 자동화된 평가 - 구성 가능한 샘플링 비율로 프로덕션 트레이스에 대해 LLM 심사를 실행
* 지속적인 품질 평가 - 사용자 경험을 방해하지 않고 실시간으로 품질 메트릭 모니터링
* 비용 효율적인 모니터링 - 커버리지와 연산 비용의 균형을 맞추는 스마트 샘플링 전략

프로덕션 모니터링을 통해 문제가 사용자에게 큰 영향을 미치기 전에 선제적으로 감지하여 해결할 수 있다는 확신을 가지고 배포할 수 있습니다.

Generative AI 모니터링에 대한 자세한 내용은 [AI Gateway 기반 추론 테이블을 사용한 모델 모니터링](https://docs.databricks.com/gcp/en/ai-gateway/inference-tables) 및 [프로덕션 품질 모니터링](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/run-scorer-in-prod) 문서를 참조하세요.

<img src="https://i.imgur.com/wv4p562.gif">

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=601859075515029&notebook=%2F05-production-monitoring%2F05.production-monitoring&demo_name=ai-agent&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fai-agent%2F05-production-monitoring%2F05.production-monitoring&version=1">

In [0]:
%pip install -U -qqqq mlflow[databricks]>=3.1.1 databricks-agents
dbutils.library.restartPython()

In [0]:
%run ../_resources/01-setup

## 프로덕션 등급 모니터 만들기

UI를 사용하거나 SDK를 직접 사용하여 모니터를 쉽게 생성할 수 있습니다:

In [0]:
from databricks.agents.monitoring import (
  AssessmentsSuiteConfig,
  GuidelinesJudge,
  create_external_monitor,
  get_external_monitor,
  update_external_monitor,
  BuiltinJudge
)
import mlflow

# 기존 실험을 재사용합니다
xp_name = os.getcwd().rsplit("/", 1)[0]+"/03-knowledge-base-rag/03.1-pdf-rag-tool"
mlflow.set_experiment(xp_name)

accuracy_guidelines = [
  """
  응답은 다음 규칙에 따라 provided_info의 모든 사실 정보를 올바르게 참조해야 합니다:
    - 모든 사실 정보는 제공된 데이터에서 직접 가져와야 하며 날조는 금지됩니다
    - 이름, 날짜, 숫자, 회사 세부 정보는 오류 없이 100% 정확해야 합니다
    - 회의 논의 사항은 데이터에 제시된 것과 동일한 감정과 우선순위로 요약되어야 합니다
    - 지원 티켓 정보는 가능한 경우 정확한 티켓 ID, 상태 및 해결 세부 정보를 포함해야 합니다
    - 모든 제품 사용 통계는 데이터에 제공된 동일한 메트릭으로 제시되어야 합니다
    - 고객 데이터에 구체적으로 언급되지 않는 한 CloudFlow 기능, 서비스 또는 제품을 참조해서는 안 됩니다
    - 데이터에 명시적으로 제공되지 않은 정보가 언급되면 자동으로 실패 처리됩니다
  """,
]

steps_and_reasoning_guildelines = [
  """
  응답은 추론 과정을 보여주지 않고 작성되어야 합니다.
    - 정보를 찾아야 한다는 언급을 하지 마세요
    - 도구나 사용된 함수를 언급하지 마세요
    - 중간 단계나 추론 과정을 말하지 마세요
  """,
]

assessments = [
  # 내장 심사
  BuiltinJudge(name="safety"),
  BuiltinJudge(name="groundedness", sample_rate=0.4),
  BuiltinJudge(name="relevance_to_query"),
  # 가이드라인은 요청과 응답을 참조할 수 있습니다
  GuidelinesJudge(guidelines={
    'accuracy': accuracy_guidelines,
    'steps_and_reasoning': steps_and_reasoning_guildelines
  })
]

In [0]:
def get_or_create_monitor():
  try:
    external_monitor = get_external_monitor(experiment_name=xp_name)
    print(f"모니터가 이미 존재합니다: {external_monitor}, 업데이트합니다")

    external_monitor = update_external_monitor(
      experiment_name=xp_name,
      assessments_config=AssessmentsSuiteConfig(
          sample=1.0,  # 샘플링 비율
          assessments=assessments
        ),
    )
    print(f"모니터가 업데이트되었습니다: {external_monitor}")

  except Exception as e:
    if "does not exist" in str(e):
      # 자동화된 프로덕션 모니터링을 위한 외부 모니터 생성
      external_monitor = create_external_monitor(
        # CREATE TABLE 권한이 있는 Unity Catalog 스키마로 변경하세요
        catalog_name=catalog,
        schema_name=dbName,
        assessments_config=AssessmentsSuiteConfig(
          sample=1.0,  # 샘플링 비율
          assessments=assessments
        )
      )
      print(f"모니터가 생성되었습니다: {external_monitor}")

In [0]:
# 모니터는 주기적으로 새로고침될 런을 생성합니다 (소량의 비용이 발생합니다)
# 실험에서 모니터를 생성하려면 주석을 해제하세요!
get_or_create_monitor()

모니터링 작업은 처음 실행될 때 약 15~30분이 소요됩니다. 초기 실행 후에는 15분마다 실행됩니다. 프로덕션 트래픽이 많은 경우 작업 완료에 추가 시간이 걸릴 수 있습니다.

작업이 실행될 때마다 다음을 수행합니다:

1. 트레이스 샘플에 대해 구성된 각 스코어러를 실행합니다
  스코어러별로 다른 샘플링 비율이 있는 경우, 모니터링 작업은 가능한 한 동일한 트레이스에 점수를 매기려고 시도합니다. 예를 들어, 스코어러 A의 샘플링 비율이 20%이고 스코어러 B가 40%인 경우, A와 B 모두에 동일한 20%의 트레이스가 사용됩니다.
2. 스코어러의 피드백을 지정된 MLflow 실험의 각 트레이스에 첨부합니다
3. 모든 트레이스(샘플링된 것뿐만 아니라)의 사본을 `trace_logs_<MLflow_experiment_id>`라는 이름의 Delta 테이블에 작성합니다
  MLflow 실험의 Trace 탭을 사용하여 모니터링 결과를 볼 수 있습니다. 또는 생성된 Delta 테이블에서 SQL 또는 Spark를 사용하여 트레이스를 쿼리할 수 있습니다.